In [1]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '1,2'
os.environ["TOKENIZERS_PARALLELISM"] = "false"

In [2]:
import lm_eval
from lm_eval.tasks import TaskManager
from lm_eval.evaluator import simple_evaluate
from lm_eval.utils import make_table

# Constants

In [3]:
TAWJEEH_DATASET_NAME = 'ArabicMMLU'
HF_EXPERIMENTAL_DATASET_NAME = 'MagedSaeed/ArabicMMLU_experimental'
TASK_NAME='NLU'
MODEL_PATH = "/raid_storage/shared_models/Meta-Llama-3.1-8B"
TUNED_MODEL_PATH = None
BATCH_SIZE=40

In [4]:
MODEL_NAME = MODEL_PATH.split('/')[-1]
TOKENIZER_PATH = MODEL_PATH

# Building the prompts dataset

In [5]:
import requests
 
from tqdm.auto import tqdm
 
prompts = None
 
tries = 10
for i in tqdm(range(tries)):
    api_response = requests.get(url='https://promptlab.up.railway.app/api/prompt/list?project_secret_key=6Wirj')
    if api_response.ok:
        prompts = api_response.json()
        break
if not prompts:
    raise Exception('Failed to fetch prompts')
prompts[:5]

  0%|          | 0/10 [00:00<?, ?it/s]

[{'id': 14869,
  'tags': [],
  'name': 'answer keyword at the end of the prompt',
  'task': {'name': 'multiple choice'},
  'status': 'APPROVED',
  'template': 'This is a question. Select the correct answer!\r\n\r\nQuestion: \r\n{{Question}}\r\n\r\nChoices:\r\n{% set choices = [A,B,C,D] %}\r\n{% for choice in choices %}\r\n{% if choice and choice.strip %}\r\n{{ answer_choices[loop.index0] }}. {{choice}}\r\n{% endif %}\r\n{% endfor %}\r\nAnswer:\r\n|||\r\n{{answer_choices[answer_choices.index(answer)]}}',
  'dataset_name': 'arbml/ArabicMMLU',
  'dataset_subset': 'default',
  'answer_choices': ['A', 'B', 'C', 'D', 'E'],
  'text_direction': 'ltr'},
 {'id': 14865,
  'tags': [],
  'name': 'QA_stance_on_topic',
  'task': {'name': 'stance detection'},
  'status': 'SUBMITTED',
  'template': "Question: If someone says the following in Arabic {{text}} about {{target}}, do you think the person is 'against', 'favor' or 'neither'? Answer: \r\n|||\r\n{{answer_choices[stance]}}",
  'dataset_name': 'ar

filter prompts:
- get only the approved ones
- get only the ones on the sarcasim detection datasets (emotone_ar,sem_eval_2018_task_1)

In [6]:
filtered_prompts = list(filter(lambda prompt: prompt['status'] == 'APPROVED' and prompt['text_direction'].lower() == 'ltr', prompts))
len(filtered_prompts)

188

### Get the dataset prompts

In [7]:
# you can either filter by task or dataset
dataset_prompts = list(
    filter(
        lambda prompt: TAWJEEH_DATASET_NAME in prompt['dataset_name'],
        filtered_prompts,
    )
)
len(dataset_prompts)

6

In [8]:
SELECTED_PROMPTS_IDS = [
    14571,
    14869,   
    14787,
    14797,
    14798,
    # 14799,
]

In [9]:
dataset_prompts = list(filter(lambda prompt: prompt['id'] in SELECTED_PROMPTS_IDS, filtered_prompts))
len(dataset_prompts)

5

### Download the dataset

In [10]:
import datasets

In [11]:
hf_exp_dataset = datasets.load_dataset(HF_EXPERIMENTAL_DATASET_NAME)
hf_exp_dataset

DatasetDict({
    train: Dataset({
        features: ['ID', 'Source', 'Country', 'Group', 'Subject', 'Level', 'Question', 'Context', 'answer', 'A', 'B', 'C', 'D', 'E', 'is_few_shot'],
        num_rows: 10000
    })
    test: Dataset({
        features: ['ID', 'Source', 'Country', 'Group', 'Subject', 'Level', 'Question', 'Context', 'answer', 'A', 'B', 'C', 'D', 'E', 'is_few_shot'],
        num_rows: 4575
    })
})

### Merge the prompts

In [12]:
from jinja2 import Environment, StrictUndefined

In [13]:
def apply_template(prompt_template, sample):
    try:
        template = prompt_template['template']
        sample['answer_choices'] = []
        for choice in prompt_template['answer_choices']:
            if choice in sample and sample[choice]:
                if isinstance(sample[choice], str) and sample[choice].strip():
                    sample['answer_choices'].append(choice)
                else:
                    sample[choice] = None
            else:
                sample[choice] = None
        # print(template)
        # print(prompt_template['id'])
        env = Environment(undefined=StrictUndefined)
        if "|||" not in template:
            raise ValueError("No ||| dividor")
        template = env.from_string(template)
        rendered_template = template.render(**sample)
        return rendered_template
    except Exception as e:
        print(prompt_template)
        print(sample)
        raise e
        

Perform generation on one example prompt, for experimentation

In [14]:
example_prompt_template = dataset_prompts[0]
print(apply_template(example_prompt_template, hf_exp_dataset['test'][1]))

This is a question. Select the correct answer!

Question: 
أول آية في سورة الفاتحة

Choices:



A. الرحمن الرحيم



B. الحمد لله رب العالمين



C. بسم الله الرحمن الرحيم




Answer:
|||
C


merge prompts

In [15]:
for prompt in dataset_prompts:
    prompt['merged_samples'] = list(
        map(
            lambda sample: apply_template(prompt, sample),
            # hf_exp_dataset['test'].select(range(100)),
            tqdm(hf_exp_dataset['test']),
        )
    )
    prompt['original_samples'] = list(hf_exp_dataset['test'])

  0%|          | 0/4575 [00:00<?, ?it/s]

  0%|          | 0/4575 [00:00<?, ?it/s]

  0%|          | 0/4575 [00:00<?, ?it/s]

  0%|          | 0/4575 [00:00<?, ?it/s]

  0%|          | 0/4575 [00:00<?, ?it/s]

# Evaluate on each prompt and report the results

In [16]:
from datasets import DatasetDict
import re

def create_hf_dataset(dataset_prompt, columns=None):
  if columns is None:
    columns = ['text', 'label','choices']
  texts = []
  labels = []
  choices = []
  for i,merged_sample in enumerate(dataset_prompt['merged_samples']):
    # prefix= merged_sample.split('|||')[0].replace('\n', ' ')
    prefix= merged_sample.split('|||')[0]
    # remove multi spaces
    # prefix = re.sub(r'\s+', ' ', prefix)
    prefix = prefix.strip()
    # add new line after prefx (this turns out to have a large effect on some prompts)
    # prefix = prefix+'\n'
    output = merged_sample.split('|||')[1].replace('\n', '')
    original_sample = dataset_prompt['original_samples'][i]
    example_choices = []
    for choice in dataset_prompt['answer_choices']:
      # if i == 1:
        # print(choice, original_sample[choice])
      if choice in original_sample and original_sample[choice] is not None:
        example_choices.append(choice)
    texts.append(prefix)
    labels.append(output)
    choices.append(example_choices)
  dataset = DatasetDict({ 'test' : datasets.Dataset.from_dict({
      columns[0]: texts,
      columns[1]: labels,
      columns[2]: choices,
  })})
  return dataset

In [17]:
dataset = create_hf_dataset(dataset_prompts[0])
dataset['test'][1]['text']

'This is a question. Select the correct answer!\n\nQuestion: \nأول آية في سورة الفاتحة\n\nChoices:\n\n\n\nA. الرحمن الرحيم\n\n\n\nB. الحمد لله رب العالمين\n\n\n\nC. بسم الله الرحمن الرحيم\n\n\n\n\nAnswer:'

In [18]:
from lm_eval.models.huggingface import HFLM
lm_obj = HFLM(
    pretrained=MODEL_PATH,
    trust_remote_code=True,
    parallelize=True,
    device_map="auto",
    tokenizer=TOKENIZER_PATH,
    batch_size=BATCH_SIZE,
)

2024-11-17:12:21:08,562 INFO     [huggingface.py:483] Using model type 'default'
2024-11-17:12:21:09,377 INFO     [huggingface.py:350] Model parallel was set to True, setting max memory per GPU to {0: 84523417600, 1: 84523417600} and device map to 'auto'


Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

In [19]:
def evaluate_tasks(tasks,dataset_sub_path=TAWJEEH_DATASET_NAME):
    # MAKE SURE THE NOTEBOOK IS RUNNING FROM THE PROJECT ROOT!
    task_manager = TaskManager(include_path=f"eval_harness_extra_tasks/{dataset_sub_path}")
    results = simple_evaluate(  # call simple_evaluate
        model=lm_obj,
        tasks=tasks,
        num_fewshot=0,
        task_manager=task_manager,
    )
    return results

In [20]:
import json

def create_and_evaluate_single_prompt(prompt, save_results=True, force_re_evaluate=False):
    prompt_id = prompt['id']
    if TUNED_MODEL_PATH:
        results_dir = f'evaluation_results/{MODEL_NAME}-tuned/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    else:
        results_dir = f'evaluation_results/{MODEL_NAME}/{TASK_NAME}/{TAWJEEH_DATASET_NAME}'
    prompt_results_file_path = f'{results_dir}/prompt_{prompt_id}.json'
    
    # Check if results exist and handle based on parameters
    if os.path.exists(prompt_results_file_path) and os.path.getsize(prompt_results_file_path) > 0:
        if not force_re_evaluate:
            print(f"Skipping prompt {prompt_id} - results already exist")
            with open(prompt_results_file_path, 'r') as f:
                prompt_results = json.load(f)
                print(make_table(prompt_results))
                return prompt_results
        else:
            print(f"Force re-evaluate enabled - reevaluating prompt {prompt_id}")
    
    # Create dataset and task files
    dataset = create_hf_dataset(prompt)
    
    # Save dataset
    dataset_dir = f'experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}'
    os.makedirs(dataset_dir, exist_ok=True)
    dataset['test'].to_parquet(f"{dataset_dir}/data.parquet")
    
    # Create YAML configuration
    yaml_text = f'''task: {TAWJEEH_DATASET_NAME}_prompt_{prompt_id}
dataset_path: experimental_hf_datasets/{TAWJEEH_DATASET_NAME}/prompt_{prompt_id}
output_type: multiple_choice
test_split: train
doc_to_text: text
doc_to_choice: choices
doc_to_target: label
metric_list:
  - metric: acc
    aggregation: mean
    higher_is_better: True
  - metric: acc_norm
    aggregation: mean
    higher_is_better: true
metadata:
  version: 1.0'''
    
    # Save YAML
    yaml_dir = f'eval_harness_extra_tasks/{TAWJEEH_DATASET_NAME}'
    os.makedirs(yaml_dir, exist_ok=True)
    with open(f'{yaml_dir}/prompt_{prompt_id}.yaml', 'w') as f:
        f.write(yaml_text)
    
    # Evaluate single prompt
    evaluation_task_name = f'{TAWJEEH_DATASET_NAME}_prompt_{prompt_id}'
    prompt_results = evaluate_tasks(tasks=[evaluation_task_name])
    
    print(make_table(prompt_results))
    
    # Save results if save_results is True
    if save_results:
        os.makedirs(results_dir, exist_ok=True)
        with open(prompt_results_file_path, 'w') as f:
            json.dump(prompt_results, f, ensure_ascii=False, indent=4, 
                     default=lambda o: '<not serializable>')
        print(f"Saved results for prompt {prompt_id}")
    else:
        print(f"Results not saved for prompt {prompt_id} (save_results=False)")
    
    print(f"Completed evaluation for prompt {prompt_id}")
    return prompt_results

In [21]:
def evaluate_all_prompts_sequentially(dataset_prompts):
    print(f"Starting sequential evaluation of {len(dataset_prompts)} prompts")
    all_results = {}
    
    for i, prompt in enumerate(dataset_prompts, 1):
        print('-' * 80)
        print(f"\nProcessing prompt {i}/{len(dataset_prompts)}")
        print("Template:", prompt['template'])
        print('-' * 80)
        
        prompt_results = create_and_evaluate_single_prompt(prompt)
        all_results[f"{TAWJEEH_DATASET_NAME}_prompt_{prompt['id']}"] = prompt_results
    
    return {'results': all_results}

In [22]:
all_results = evaluate_all_prompts_sequentially(dataset_prompts=dataset_prompts)

Starting sequential evaluation of 5 prompts
--------------------------------------------------------------------------------

Processing prompt 1/5
Template: This is a question. Select the correct answer!

Question: 
{{Question}}

Choices:
{% set choices = [A,B,C,D] %}
{% for choice in choices %}
{% if choice and choice.strip %}
{{ answer_choices[loop.index0] }}. {{choice}}
{% endif %}
{% endfor %}
Answer:
|||
{{answer_choices[answer_choices.index(answer)]}}
--------------------------------------------------------------------------------
Skipping prompt 14869 - results already exist
|         Tasks         |Version|Filter|n-shot| Metric |   |Value |   |Stderr|
|-----------------------|------:|------|-----:|--------|---|-----:|---|-----:|
|ArabicMMLU_prompt_14869|      1|none  |     0|acc     |↑  |0.4874|±  |0.0074|
|                       |       |none  |     0|acc_norm|↑  |0.4874|±  |0.0074|

--------------------------------------------------------------------------------

Processing 

In [ ]:
exit()

: 